# CNN Training Pipeline

End-to-end notebook covering every CNN process in `models/CNN/`:

0. Setup
1. Baseline student training (one or all)
2. Teacher training (one or all)
3. Knowledge distillation (one combo or all 15)
4. Optuna HPO (one student or all)
5. Optuna HPO with knowledge distillation (one combo or all 15)
6. Inference / sanity-check on a saved model
7. Evaluation via the team-standard `evaluation/` module (one model or all)

Each section has a small **Configure** cell at the top (constants to change) and a **Run** cell underneath. Re-run cells freely — outputs land in `models/CNN/checkpoints/` and `models/CNN/evaluations/`.

## 0. Setup

Adds the repo root and the CNN folder to `sys.path` (so peer modules like `datasets.loaders` and `evaluation.metrics` import cleanly), switches the working directory to the repo root, and loads everything we'll need.

In [ ]:
import sys, os, copy, json, time
from pathlib import Path

_here = Path.cwd()
while not (_here / "datasets").exists() and _here != _here.parent:
    _here = _here.parent
REPO_ROOT = _here
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "models" / "CNN"))

import torch
import torch.nn as nn
import numpy as np

from main import Config, set_seed, get_loaders, _summary_metrics, train_student, train_teacher, train_student_with_kd
from students import CNN_Nano, CNN_Micro, CNN_Base, CNN_Large, CNN_XLarge
from teachers import ResNet50_1D, ResNet101_1D, ResNet152_1D
from training import train_model, evaluate
from distillation import distillation_loss
from evaluation.metrics import compute_multilabel_metrics, compute_curves
from evaluation.plots import save_metric_curves, save_confusion_matrices

STUDENT_CLS = {"nano": CNN_Nano, "micro": CNN_Micro, "base": CNN_Base, "large": CNN_Large, "xlarge": CNN_XLarge}
TEACHER_CLS = {"resnet50": ResNet50_1D, "resnet101": ResNet101_1D, "resnet152": ResNet152_1D}
STUDENT_NAMES = list(STUDENT_CLS.keys())
TEACHER_NAMES = list(TEACHER_CLS.keys())

CKPT_DIR = Path("models/CNN/checkpoints")
EVAL_DIR = Path("models/CNN/evaluations")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"repo root: {REPO_ROOT}")
print(f"device:    {DEVICE}")

## 1. Train students

Calls `train_student(cfg)` from `main.py`. Saves the best-val-acc checkpoint to `models/CNN/checkpoints/student_<name>.pth` and prints test-set metrics at the end.

### 1.1 Train one student

In [ ]:
# === Configure ===
STUDENT = "nano"     # one of: nano, micro, base, large, xlarge
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
cfg = Config(mode="student", student_name=STUDENT, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
train_student(cfg)

### 1.2 Train all students (sequential)

In [ ]:
# === Configure ===
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
for name in STUDENT_NAMES:
    print(f"\n========== {name.upper()} ==========")
    cfg = Config(mode="student", student_name=name, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
    train_student(cfg)

## 2. Train teachers

1D ResNet teachers. Saves to `models/CNN/checkpoints/teacher_<name>.pth`. Teachers are bigger than students — expect ~6 s/epoch on a modern GPU.

### 2.1 Train one teacher

In [ ]:
# === Configure ===
TEACHER = "resnet50"  # resnet50 | resnet101 | resnet152
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
cfg = Config(mode="teacher", teacher_name=TEACHER, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
train_teacher(cfg)

### 2.2 Train all teachers (sequential)

In [ ]:
# === Configure ===
EPOCHS = 100
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
for name in TEACHER_NAMES:
    print(f"\n========== {name.upper()} ==========")
    cfg = Config(mode="teacher", teacher_name=name, epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, device=DEVICE)
    train_teacher(cfg)

## 3. Knowledge distillation

Single-step KD: student trained against teacher logits. If a `teacher_<name>.pth` exists in `checkpoints/`, it's loaded; otherwise the teacher is pretrained on the fly for `teacher_epochs`. Saves to `student_kd_<student>_from_<teacher>.pth`.

### 3.1 Train one student with one teacher

In [ ]:
# === Configure ===
STUDENT = "nano"
TEACHER = "resnet50"
EPOCHS = 100
TEACHER_PRETRAIN_EPOCHS = 30   # only used if teacher_<name>.pth is missing
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.5
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
teacher_ckpt = CKPT_DIR / f"teacher_{TEACHER}.pth"
cfg = Config(
    mode="student_kd", student_name=STUDENT, teacher_name=TEACHER,
    epochs=EPOCHS, teacher_epochs=TEACHER_PRETRAIN_EPOCHS,
    kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
    batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
    teacher_ckpt=str(teacher_ckpt) if teacher_ckpt.exists() else None,
)
train_student_with_kd(cfg)

### 3.2 Train all 15 student-teacher KD combinations

Loops over every `(student, teacher)` pair. If a teacher checkpoint isn't on disk yet, it's pretrained the first time it's needed — subsequent pairs that share the same teacher reuse the saved file.

In [ ]:
# === Configure ===
EPOCHS = 100
TEACHER_PRETRAIN_EPOCHS = 30
KD_TEMPERATURE = 4.0
KD_ALPHA = 0.5
BATCH_SIZE = 32
LR = 1e-3

# === Run ===
for teacher_name in TEACHER_NAMES:
    teacher_ckpt = CKPT_DIR / f"teacher_{teacher_name}.pth"
    for student_name in STUDENT_NAMES:
        print(f"\n========== {student_name.upper()} from {teacher_name.upper()} ==========")
        cfg = Config(
            mode="student_kd", student_name=student_name, teacher_name=teacher_name,
            epochs=EPOCHS, teacher_epochs=TEACHER_PRETRAIN_EPOCHS,
            kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
            batch_size=BATCH_SIZE, lr=LR, device=DEVICE,
            teacher_ckpt=str(teacher_ckpt) if teacher_ckpt.exists() else None,
        )
        train_student_with_kd(cfg)

## 4. Optuna HPO (no KD)

Calls `optimization.run` which sweeps architecture + training hyperparams per student size. Each trial trains up to `trial_epochs` with patience-based early stopping; trials scored by `val_accuracy × speed_penalty` against a 1-epoch baseline.

Outputs per size: `optuna_<size>.pth` (with `arch_kwargs`) and `optuna_<size>.txt` (human-readable summary).

### 4.1 HPO for one student

In [ ]:
from optimization import HPOConfig, run as run_hpo

# === Configure ===
STUDENT = "nano"
N_TRIALS = 50
TRIAL_EPOCHS = 100
EARLY_STOP_PATIENCE = 5
EARLY_STOP_MIN_DELTA = 2.0

# === Run ===
hcfg = HPOConfig(
    sizes=[STUDENT], n_trials_per_size=N_TRIALS, trial_epochs=TRIAL_EPOCHS,
    early_stop_patience=EARLY_STOP_PATIENCE, early_stop_min_delta=EARLY_STOP_MIN_DELTA,
    device=DEVICE,
)
run_hpo(hcfg)

### 4.2 HPO for all students

In [ ]:
from optimization import HPOConfig, run as run_hpo

# === Configure ===
N_TRIALS = 50
TRIAL_EPOCHS = 100

# === Run ===
hcfg = HPOConfig(n_trials_per_size=N_TRIALS, trial_epochs=TRIAL_EPOCHS, device=DEVICE)
run_hpo(hcfg)

## 5. Optuna HPO with knowledge distillation

Same scoring shape (`val_accuracy × speed_penalty`) but the student is trained against teacher logits instead of plain BCE. Tunes architecture, `lr`, `weight_decay`, `batch_size`, `kd_temperature`, and `kd_alpha`. The teacher is loaded once per study (from `teacher_<name>.pth`, or pretrained on the fly if missing) and reused across trials.

Outputs per combo: `optuna_kd_<student>_from_<teacher>.pth` and `optuna_kd_<student>_from_<teacher>.txt`.

In [ ]:
# === KD HPO helper ===
# Reuses the search space, baseline timing, latency measurement, and penalty from optimization.py;
# differs only in that the per-trial loop trains against the teacher.
import optuna
from optimization import (SEARCH_SPACES, TRAIN_SPACE, suggest_arch, measure_baseline,
                          measure_latency, speed_penalty)

def _train_kd_trial(student, teacher, train_loader, val_loader, lr, wd, T, alpha,
                    epochs, device, trial, patience, min_delta):
    optimizer = torch.optim.Adam(student.parameters(), lr=lr, weight_decay=wd)
    eval_criterion = nn.BCEWithLogitsLoss()
    teacher.eval()
    best_acc, best_state, last_milestone, stall = -1.0, None, -float("inf"), 0
    epoch = 0
    for epoch in range(epochs):
        student.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            s_logits = student(x)
            with torch.no_grad():
                t_logits = teacher(x)
            loss = distillation_loss(s_logits, t_logits, y, T, alpha)
            loss.backward(); optimizer.step()
        val = evaluate(student, val_loader, eval_criterion, device)
        acc = val["accuracy"]
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(student.state_dict())
        if acc >= last_milestone + min_delta:
            last_milestone, stall = acc, 0
        else:
            stall += 1
        if stall >= patience: break
        trial.report(acc, epoch)
        if trial.should_prune(): raise optuna.TrialPruned()
    return best_acc, best_state, epoch + 1

def _load_or_pretrain_teacher(teacher_name, pretrain_epochs, device):
    teacher = TEACHER_CLS[teacher_name]().to(device)
    ckpt = CKPT_DIR / f"teacher_{teacher_name}.pth"
    if ckpt.exists():
        teacher.load_state_dict(torch.load(ckpt, map_location=device))
        print(f"[teacher] loaded {ckpt}")
    else:
        print(f"[teacher] no checkpoint, pretraining {teacher_name} for {pretrain_epochs} epochs")
        cfg = Config(mode="teacher", teacher_name=teacher_name, epochs=pretrain_epochs, device=device)
        teacher = train_teacher(cfg).to(device)
    teacher.eval()
    return teacher

def kd_hpo(student_name, teacher_name, n_trials=30, trial_epochs=60,
           teacher_pretrain_epochs=30, patience=5, min_delta=2.0, seed=42):
    device = torch.device(DEVICE)
    teacher = _load_or_pretrain_teacher(teacher_name, teacher_pretrain_epochs, device)

    # Baseline timing: default student arch trained 1 epoch with KD (so reference is comparable)
    set_seed(seed)
    base_model = STUDENT_CLS[student_name]().to(device)
    train_loader, val_loader, _ = get_loaders(Config(batch_size=32, seed=seed, device=DEVICE))
    base_opt = torch.optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-4)
    base_model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        base_opt.zero_grad()
        with torch.no_grad(): t = teacher(x)
        loss = distillation_loss(base_model(x), t, y, 4.0, 0.5)
        loss.backward(); base_opt.step()
    base_latency = measure_latency(base_model, val_loader, device)
    base_val_acc = evaluate(base_model, val_loader, nn.BCEWithLogitsLoss(), device)["accuracy"]
    print(f"[baseline] latency={base_latency:.3f}ms  val_acc={base_val_acc:.2f}%")

    best = {"score": -1.0}
    def objective(trial):
        set_seed(seed + trial.number)
        arch = suggest_arch(trial, student_name)
        lr  = trial.suggest_float("lr",           *TRAIN_SPACE["lr"],           log=True)
        wd  = trial.suggest_float("weight_decay", *TRAIN_SPACE["weight_decay"], log=True)
        bs  = trial.suggest_categorical("batch_size", TRAIN_SPACE["batch_size"])
        T   = trial.suggest_float("kd_temperature", 1.0, 10.0)
        a   = trial.suggest_float("kd_alpha",        0.1, 0.9)

        tl, vl, _ = get_loaders(Config(batch_size=bs, seed=seed, device=DEVICE))
        student = STUDENT_CLS[student_name](**arch).to(device)
        best_acc, best_state, epochs_run = _train_kd_trial(
            student, teacher, tl, vl, lr, wd, T, a, trial_epochs, device, trial, patience, min_delta,
        )
        student.load_state_dict(best_state)
        latency = measure_latency(student, vl, device)
        ratio = latency / base_latency
        score = (best_acc / 100.0) * speed_penalty(ratio)
        train_kw = {"lr": lr, "weight_decay": wd, "batch_size": bs, "kd_temperature": T, "kd_alpha": a}
        if score > best["score"]:
            best.update(score=score, state=best_state, arch_kwargs=arch, train_kwargs=train_kw,
                        val_accuracy=best_acc, latency_ms=latency, ratio=ratio, epochs_run=epochs_run)
        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    )
    study.optimize(objective, n_trials=n_trials)

    pth = CKPT_DIR / f"optuna_kd_{student_name}_from_{teacher_name}.pth"
    torch.save({"state_dict": best["state"], "arch_kwargs": best["arch_kwargs"]}, pth)
    txt = CKPT_DIR / f"optuna_kd_{student_name}_from_{teacher_name}.txt"
    arch_repr = ", ".join(f"{k}={v!r}" for k, v in best["arch_kwargs"].items())
    txt.write_text(
        f"=== optuna_kd_{student_name}_from_{teacher_name} ===\n"
        f"trials run:         {len(study.trials)}\n"
        f"best score:         {best['score']:.4f}\n"
        f"best val accuracy:  {best['val_accuracy']:.2f}%\n"
        f"inference latency:  {best['latency_ms']:.3f} ms (baseline {base_latency:.3f}, ratio {best['ratio']:.3f}x)\n"
        f"epochs trained:     {best['epochs_run']} / {trial_epochs}\n\n"
        f"Arch kwargs:\n" + "\n".join(f"  {k:<14} = {v}" for k, v in best["arch_kwargs"].items()) + "\n\n"
        f"Training kwargs:\n" + "\n".join(f"  {k:<14} = {v}" for k, v in best["train_kwargs"].items()) + "\n"
    )
    print(f"[done] score={best['score']:.4f} acc={best['val_accuracy']:.2f}% latency={best['latency_ms']:.3f}ms -> {pth.name}")
    return best

### 5.1 KD HPO for one student-teacher combo

In [ ]:
# === Configure ===
STUDENT = "nano"
TEACHER = "resnet50"
N_TRIALS = 30
TRIAL_EPOCHS = 60

# === Run ===
kd_hpo(STUDENT, TEACHER, n_trials=N_TRIALS, trial_epochs=TRIAL_EPOCHS)

### 5.2 KD HPO for all 15 student-teacher combos

Heavy: 15 studies × `N_TRIALS` trials × variable epochs (with early stop). Defaults below assume an overnight run. Cut `N_TRIALS` or `TRIAL_EPOCHS` for quicker exploration.

In [ ]:
# === Configure ===
N_TRIALS = 20
TRIAL_EPOCHS = 50

# === Run ===
for teacher_name in TEACHER_NAMES:
    for student_name in STUDENT_NAMES:
        print(f"\n========== KD HPO: {student_name.upper()} from {teacher_name.upper()} ==========")
        kd_hpo(student_name, teacher_name, n_trials=N_TRIALS, trial_epochs=TRIAL_EPOCHS)

## 6. Inference / sanity check

Pick any checkpoint in `models/CNN/checkpoints/` and verify it loads and produces sensible predictions on a test batch. Filename convention drives which model class is rebuilt:

- `student_<size>.pth` → `STUDENT_CLS[<size>]()`
- `teacher_<name>.pth` → `TEACHER_CLS[<name>]()`
- `student_kd_<student>_from_<teacher>.pth` → `STUDENT_CLS[<student>]()`
- `optuna_<size>.pth` and `optuna_kd_<student>_from_<teacher>.pth` → `STUDENT_CLS[<...>](**arch_kwargs)`

In [ ]:
def load_checkpoint(ckpt_path):
    p = Path(ckpt_path)
    name = p.stem
    obj = torch.load(p, map_location=DEVICE, weights_only=False)
    arch_kwargs = {}
    if isinstance(obj, dict) and "state_dict" in obj:
        state, arch_kwargs = obj["state_dict"], obj.get("arch_kwargs", {})
    else:
        state = obj
    if name.startswith("optuna_kd_"):
        size = name[len("optuna_kd_"):].split("_from_")[0]
        model = STUDENT_CLS[size](**arch_kwargs)
    elif name.startswith("optuna_"):
        size = name[len("optuna_"):]
        model = STUDENT_CLS[size](**arch_kwargs)
    elif name.startswith("student_kd_"):
        size = name[len("student_kd_"):].split("_from_")[0]
        model = STUDENT_CLS[size]()
    elif name.startswith("student_"):
        model = STUDENT_CLS[name[len("student_"):]]()
    elif name.startswith("teacher_"):
        model = TEACHER_CLS[name[len("teacher_"):]]()
    else:
        raise ValueError(f"Unrecognized checkpoint name pattern: {name}")
    model.load_state_dict(state)
    return model.to(DEVICE).eval(), arch_kwargs

In [ ]:
# === Configure ===
CHECKPOINT = CKPT_DIR / "student_nano.pth"   # change to any .pth in checkpoints/

# === Run ===
model, arch = load_checkpoint(CHECKPOINT)
print(f"loaded {CHECKPOINT.name}  params={sum(p.numel() for p in model.parameters()):,}")
if arch: print(f"arch_kwargs={arch}")

_, _, test_loader = get_loaders(Config(batch_size=8, device=DEVICE))
x, y = next(iter(test_loader))
x, y = x.to(DEVICE), y.to(DEVICE)
with torch.no_grad():
    probs = torch.sigmoid(model(x))
preds = (probs > 0.5).float()
for i in range(min(5, x.shape[0])):
    print(f"  sample {i}: true={y[i].tolist()}  pred={preds[i].tolist()}  probs={[f'{p:.2f}' for p in probs[i].tolist()]}")

## 7. Evaluation

Runs each model on the test split and computes the team-standard multilabel metrics (`accuracy`, `f1_macro`, `auprc_macro`, `auroc_macro`, per-finger breakdowns) plus PR/ROC curves and per-finger confusion matrices using the peer's `evaluation/` module. Results land in `models/CNN/evaluations/<model_name>/`.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, device):
    model.eval()
    ps, ys = [], []
    for x, y in loader:
        ps.append(torch.sigmoid(model(x.to(device))).cpu())
        ys.append(y)
    return torch.cat(ps), torch.cat(ys)

def evaluate_checkpoint(ckpt_path, out_dir=None, save_per_finger=True):
    ckpt_path = Path(ckpt_path)
    name = ckpt_path.stem
    out_dir = Path(out_dir) if out_dir else EVAL_DIR / name
    out_dir.mkdir(parents=True, exist_ok=True)

    model, _ = load_checkpoint(ckpt_path)
    _, _, test_loader = get_loaders(Config(batch_size=64, device=DEVICE))
    probs, targets = collect_predictions(model, test_loader, torch.device(DEVICE))

    metrics = compute_multilabel_metrics(probs, targets)
    curves = compute_curves(probs, targets)
    save_metric_curves(curves, str(out_dir), save_per_finger=save_per_finger)
    save_confusion_matrices(probs, targets, str(out_dir))
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    print(f"[{name}] acc={metrics['accuracy']:.3f}  f1={metrics['f1_macro']:.3f}  "
          f"auprc={metrics['auprc_macro']:.3f}  auroc={metrics['auroc_macro']:.3f}  -> {out_dir}")
    return metrics

### 7.1 Evaluate one model

In [ ]:
# === Configure ===
CHECKPOINT = CKPT_DIR / "student_nano.pth"

# === Run ===
evaluate_checkpoint(CHECKPOINT)

### 7.2 Evaluate every checkpoint in `checkpoints/`

Iterates all `.pth` files, evaluates each, and writes a top-level `summary.json` ranking them by `auprc_macro` (a sensible robust default for multi-label).

In [ ]:
ckpts = sorted(CKPT_DIR.glob("*.pth"))
results = {}
for p in ckpts:
    try:
        results[p.stem] = evaluate_checkpoint(p)
    except Exception as e:
        print(f"[skip] {p.name}: {e}")

ranked = sorted(results.items(), key=lambda kv: kv[1].get("auprc_macro", float("-inf")), reverse=True)
summary = {name: {k: m.get(k) for k in ("accuracy", "f1_macro", "auprc_macro", "auroc_macro")} for name, m in ranked}
(EVAL_DIR / "summary.json").write_text(json.dumps(summary, indent=2))

print("\n========== RANKED BY auprc_macro ==========")
print(f"{'model':<48} {'acc':>6} {'f1':>6} {'auprc':>7} {'auroc':>7}")
for name, m in ranked:
    print(f"{name:<48} {m['accuracy']:>6.3f} {m['f1_macro']:>6.3f} {m['auprc_macro']:>7.3f} {m['auroc_macro']:>7.3f}")